In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output
import country_converter as coco
import re
import numpy as np
import time

# ---  Load  data ---
file_path = "Scale_up_output_MS.csv"
df = pd.read_csv(file_path)

# Rename first column 
df.rename(columns={df.columns[0]: "Country"}, inplace=True)

# --- Safe cleaning function ---
def clean_value(x):
    if isinstance(x, str):
        # Case: (2.0+/-1.1)e+05
        match = re.match(r"\(([\d\.]+)\+/-.*?\)(e[+-]?\d+)?", x)
        if match:
            base = match.group(1)
            exp = match.group(2) if match.group(2) else ""
            try:
                return float(base + exp)
            except:
                return np.nan
        # Case: 0.0+/-0
        match2 = re.match(r"([\d\.]+)\+/-", x)
        if match2:
            try:
                return float(match2.group(1))
            except:
                return np.nan
        # Case: plain number
        try:
            return float(x)
        except:
            return np.nan
    return x

week_cols = df.columns[2:]  
df[week_cols] = df[week_cols].applymap(clean_value)

df = df.copy()

# --- Accumulate airflow across weeks ---
df[week_cols] = df[week_cols].cumsum(axis=1)

# --- Add UN region + subregion + ISO3 ---
cc = coco.CountryConverter()
df["Region"] = cc.convert(names=df["Country"], to="UNregion")
df["Subregion"] = cc.convert(names=df["Country"], to="continent")  # fallback
df["ISO3"] = cc.convert(names=df["Country"], to="ISO3")

df_melt = df.melt(
    id_vars=["Country", "ISO3", "Region", "Subregion"],
    value_vars=week_cols,
    var_name="Week", value_name="AirFlow"
)

# --- color scale ---
vmin = df[week_cols].min().min()
vmax = df[week_cols].max().max()

# ---  Widgets ---
week_index = widgets.IntSlider(
    value=1,
    min=1,
    max=len(week_cols),
    step=1,
    description="Week:"
)

group_choice = widgets.Dropdown(
    options=["Region", "Subregion"],
    value="Region",
    description="Group by:"
)

region_filter = widgets.Dropdown(
    options=["All"] 
            + sorted(df["Region"].dropna().unique().tolist()) 
            + sorted(df["Subregion"].dropna().unique().tolist()),
    value="All",
    description="Filter:"
)

play_button = widgets.Button(
    description="▶ Play",
    tooltip="Auto-play weeks",
    button_style="success"
)

out = widgets.Output()

# --- Plotting ---
def update_plots(change=None):
    week = str(week_index.value)
    group = group_choice.value
    filter_value = region_filter.value
    
    with out:
        clear_output(wait=True)

        # Filter dataframe if region filter applied
        if filter_value != "All":
            df_filtered = df[(df["Region"] == filter_value) | (df["Subregion"] == filter_value)]
        else:
            df_filtered = df

        # Choropleth for cumulative airflow
        fig1 = px.choropleth(
            df_filtered,
            locations="ISO3",          
            locationmode="ISO-3",
            color=week,
            hover_name="Country",
            color_continuous_scale=[(0, "white"), (1, "darkgreen")],
            range_color=[vmin, vmax],
            title=f"Total Air Flow by Country (up to Week {week})"
        )
        fig1.show()

        # Historical line plot 
        fig2 = px.line(
            df_melt,
            x="Week", y="AirFlow", color=group,
            title=f"Total Air Flow by UN {group}"
        )
        fig2.update_layout(
            plot_bgcolor="#E6FFE6",   # chart area
        )
        fig2.show()

# ---  button logic ---
def play_clicked(b):
    for w in range(1, len(week_cols) + 1):
        week_index.value = w
        time.sleep(0.5)  # adjust playback speed 

play_button.on_click(play_clicked)


week_index.observe(update_plots, names="value")
group_choice.observe(update_plots, names="value")
region_filter.observe(update_plots, names="value")

from IPython.display import HTML
from ipywidgets import VBox

custom_style = """
<style>
#custom-container {
    background-color: #fffacd;   /* light yellow */
    padding: 15px;
    border-radius: 10px;
    margin-top: 10px;
    margin-bottom: 20px;
}
#custom-banner {
    width: 100%;
    background-color: #006400;   /* dark green */
    color: white;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    padding: 10px;
    border-radius: 6px;
    margin-bottom: 15px;
}
</style>
"""
display(HTML(custom_style))


container = widgets.VBox([
    widgets.HTML('<div id="custom-banner">ALLFED</div>'),
    week_index,
    group_choice,
    region_filter,
    play_button,
    out
])
container.layout = widgets.Layout(width="100%", padding="15px", border="solid 2px #006400", border_radius="10px", background_color="#fffacd")

display(container)


update_plots()


C:\Users\26959\AppData\Local\Temp\ipykernel_24012\2673749129.py:44: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[week_cols] = df[week_cols].applymap(clean_value)
C:\Users\26959\AppData\Local\Temp\ipykernel_24012\2673749129.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Region"] = cc.convert(names=df["Country"], to="UNregion")
C:\Users\26959\AppData\Local\Temp\ipykernel_24012\2673749129.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Subregion"] = cc.convert(names=df["C

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import country_converter as coco
import numpy as np
import time
import re
import matplotlib.pyplot as plt
from uncertainties import ufloat
from uncertainties import unumpy as unp
from ipywidgets import VBox

# --- Safe parse function for uncertainties ---
def parse_ufloat(val):
    """Convert string like '(8+/-4)e+06' into a ufloat, else plain float."""
    try:
        if isinstance(val, (int, float)):
            return float(val)  # keep plain float if no error
        if isinstance(val, str):
            val = val.strip()
            if val in ["", "0", "0.0", "0.0+/-0.0", "0+/-0"]:
                return 0.0
            if "+/-" in val:
                base, exp = val.split("e") if "e" in val else (val, "0")
                val_, err = base.strip("()").split("+/-")
                return ufloat(float(val_) * 10**int(exp), float(err) * 10**int(exp))
            return float(val)
    except Exception:
        return np.nan
    return val


# --- Aggregation function ---
def aggregate_by_region(df, region_type="UNregion"):
    """
    Aggregate countries into regions using country_converter.
    region_type can be 'UNregion', 'UNcontinent', etc.
    """
    cc = coco.CountryConverter()
    df = df.copy()

    # Add region column
    df["Region"] = cc.convert(names=df["Country"].tolist(), to=region_type)

    # Drop unmapped countries
    df = df[df["Region"] != "not found"]

    region_data = {}
    for region, group in df.groupby("Region"):
        numeric = group.drop(columns=["Country", "Region"])
        numeric = numeric.map(parse_ufloat)
        summed = numeric.apply(lambda col: sum([v for v in col if v is not pd.NA]), axis=0)
        region_data[region] = summed

    region_df = pd.DataFrame(region_data).T
    region_df = region_df.reset_index().rename(columns={"index": "Region"})
    return region_df

# --- Load data ---
file_path = "Scale_up_output_MS.csv"
df = pd.read_csv(file_path)

# Rename first column 
df.rename(columns={df.columns[0]: "Country"}, inplace=True)

week_cols = df.columns[2:]  

# --- Normal dataframe (for choropleth) ---
def clean_value(x):
    if isinstance(x, str):
        match = re.match(r"\(([\d\.]+)\+/-.*?\)(e[+-]?\d+)?", x)
        if match:
            base = match.group(1)
            exp = match.group(2) if match.group(2) else ""
            try:
                return float(base + exp)
            except:
                return np.nan
        match2 = re.match(r"([\d\.]+)\+/-", x)
        if match2:
            try:
                return float(match2.group(1))
            except:
                return np.nan
        try:
            return float(x)
        except:
            return np.nan
    return x

df[week_cols] = df[week_cols].applymap(clean_value)
df = df.copy()
df[week_cols] = df[week_cols].cumsum(axis=1)

# --- Uncertainty dataframe ---
df_u = pd.read_csv(file_path)
df_u.rename(columns={df_u.columns[0]: "Country"}, inplace=True)
for col in week_cols:
    df_u[col] = df_u[col].apply(parse_ufloat)
df_u = df_u.copy()

# --- Add regions ---
cc = coco.CountryConverter()
df["Region"] = cc.convert(names=df["Country"], to="UNregion")
df["Subregion"] = cc.convert(names=df["Country"], to="continent")
df["ISO3"] = cc.convert(names=df["Country"], to="ISO3")

df_u["Region"] = cc.convert(names=df_u["Country"], to="UNregion")
df_u["Subregion"] = cc.convert(names=df_u["Country"], to="continent")
df_u["ISO3"] = cc.convert(names=df_u["Country"], to="ISO3")

# --- color scale ---
vmin = df[week_cols].min().min()
vmax = df[week_cols].max().max()

# --- Widgets ---
week_index = widgets.IntSlider(
    value=1,
    min=1,
    max=len(week_cols),
    step=1,
    description="Week:"
)

group_choice = widgets.Dropdown(
    options=["Region", "Subregion"],
    value="Region",
    description="Group by:"
)

region_filter = widgets.Dropdown(
    options=["All"] 
            + sorted(df["Region"].dropna().unique().tolist()) 
            + sorted(df["Subregion"].dropna().unique().tolist()),
    value="All",
    description="Filter:"
)

play_button = widgets.Button(
    description="▶ Play",
    tooltip="Auto-play weeks",
    button_style="success"
)

# NEW: Multi-region selector for uncertainty plot
region_select = widgets.SelectMultiple(
    options=sorted(df_u["Region"].dropna().unique().tolist()),
    value=["Eastern Asia"],   # default
    description="Regions:",
    layout=widgets.Layout(width="50%", height="150px")
)

out = widgets.Output()

# --- Plotting ---
def update_plots(change=None):
    week = str(week_index.value)
    filter_value = region_filter.value
    
    with out:
        clear_output(wait=True)

        # Filter dataframe if region filter applied
        if filter_value != "All":
            df_filtered = df[(df["Region"] == filter_value) | (df["Subregion"] == filter_value)]
        else:
            df_filtered = df

        # Choropleth for cumulative airflow
        fig1 = px.choropleth(
            df_filtered,
            locations="ISO3",          
            locationmode="ISO-3",
            color=week,
            hover_name="Country",
            color_continuous_scale=[(0, "white"), (1, "darkgreen")],
            range_color=[vmin, vmax],
            title=f"Total Air Flow by Country (up to Week {week})"
        )
        fig1.show()

        # Uncertainty line plot (region-level aggregation!)
        region_df = aggregate_by_region(df_u, region_type="UNregion")
        fig, ax = plt.subplots(figsize=(10,6))
        for region in region_select.value:
            subdf = region_df[region_df["Region"] == region]
            if subdf.empty:
                continue
            series = subdf.iloc[0, 1:].values.astype(object)
            vals = unp.nominal_values(series)
            errs = unp.std_devs(series)
            t = np.arange(len(vals))
            ax.plot(t, vals, label=region)
            ax.fill_between(t, vals-errs, vals+errs, alpha=0.2)
        ax.set_title("Total Air Flow by Selected UN Regions (with uncertainties)")
        ax.set_xlabel("Weeks")
        ax.set_ylabel("AirFlow")
        ax.legend(loc="upper left", bbox_to_anchor=(1,1))
        plt.show()

# --- button logic ---
def play_clicked(b):
    for w in range(1, len(week_cols) + 1):
        week_index.value = w
        time.sleep(0.5)  # adjust playback speed 

play_button.on_click(play_clicked)

week_index.observe(update_plots, names="value")
group_choice.observe(update_plots, names="value")
region_filter.observe(update_plots, names="value")
region_select.observe(update_plots, names="value")

# --- Custom ALLFED banner ---
custom_style = """
<style>
#custom-container {
    background-color: #fffacd;   /* light yellow */
    padding: 15px;
    border-radius: 10px;
    margin-top: 10px;
    margin-bottom: 20px;
}
#custom-banner {
    width: 100%;
    background-color: #006400;   /* dark green */
    color: white;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    padding: 10px;
    border-radius: 6px;
    margin-bottom: 15px;
}
</style>
"""
display(HTML(custom_style))

container = VBox([
    widgets.HTML('<div id="custom-banner">ALLFED</div>'),
    week_index,
    group_choice,
    region_filter,
    play_button,
    region_select,   # NEW selector box for uncertainty plot
    out
])
container.layout = widgets.Layout(width="100%", padding="15px", border="solid 2px #006400", border_radius="10px", background_color="#fffacd")

display(container)

update_plots()


C:\Users\26959\AppData\Local\Temp\ipykernel_24012\3682739797.py:91: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.

c:\Users\26959\AppData\Local\Programs\Python\Python313\Lib\site-packages\uncertainties\core.py:1024: UserWarning:

Using UFloat objects with std_dev==0 may give unexpected results.

C:\Users\26959\AppData\Local\Temp\ipykernel_24012\3682739797.py:104: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\26959\AppData\Local\Temp\ipykernel_24012\3682739797.py:105: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, u